# RSNA Knee — rsna-knee-ens-cnx-tta3c

Internet off. La session a **deux T4** : le CNX sur cuda:0, le ViT et le distil sur cuda:1.
Pas de notebook public. Pas de hflip, pas de zoom.

**Baseline 60 %** : `cnx_r7_in22k_v3b_oofmask_full` SWA 7-10 (gold 0.976).
Entree 224, FOV 130 mm. ConvNeXt en float32.
**TTA x4, quatre passages, sans l'identite** : rotation +6 deg, rotation -6 deg,
contraste 1.08, luminosite +0.05. Meme recette sur le CNX, le ViT et le distil.
Pas de hflip, pas de zoom, pas d'echange de labels. Moyenne des quatre sigmoides.

**40 % specialistes soft** (porte 0.015, softmax T=0.05). Residu d'arrondi
sur le CNX.

- ViT-k320 : `vit_k_llrd08_320_v3b_oofmask_v20h_full_oai3` SWA 1-3.
  Entree 320, FOV 130 mm. **Rotation +/-6, contraste 1.08, luminosite +0.05.**
- v20h distil : `v20h_crop145_dino_ep11_v3b_rsna_distil_main_full` SWA 7-9.
  Entree 224, FOV 145 mm. **Meme TTA x4**, sans hflip.


In [ ]:
import os
from pathlib import Path

KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    DATA_DIR = Path('/kaggle/input')
    print('contenu /kaggle/input :', sorted(p.name for p in DATA_DIR.iterdir()))
    COMP_DIR = None
    for base in [DATA_DIR / 'competitions', DATA_DIR]:
        if base.exists():
            for p in sorted(base.iterdir()):
                if (p / 'test.csv').exists():
                    COMP_DIR = p
                    break
        if COMP_DIR is not None:
            break
    if COMP_DIR is None:
        raise FileNotFoundError('test.csv introuvable sous /kaggle/input')
    search_roots = []
    if (DATA_DIR / 'datasets').exists():
        search_roots.append(DATA_DIR / 'datasets')
    search_roots.append(DATA_DIR)

    def _find_weight(name):
        for root in search_roots:
            hits = list(root.rglob(name))
            if hits:
                return hits[0]
        raise FileNotFoundError(name)

    WEIGHTS_VIT = _find_weight('vit_k320_oai3_swa_ep1_3.pt')
    WEIGHTS_V20H = _find_weight('v20h_crop145_distil_swa_ep7_9.pt')
    WEIGHTS_CNX = _find_weight('cnx_r7_swa_ep7_10.pt')
else:
    COMP_DIR = Path('../data/csv')
    WEIGHTS_VIT = Path('kaggle_dataset_vit_k/_upload_k320_oai3_swa/vit_k320_oai3_swa_ep1_3.pt')
    WEIGHTS_V20H = Path('kaggle_dataset_v20h_lat/_upload_crop145_distil_swa789/v20h_crop145_distil_swa_ep7_9.pt')
    WEIGHTS_CNX = Path('kaggle_dataset_cnx/_upload_swa78910/cnx_r7_swa_ep7_10.pt')

TEST_CSV = COMP_DIR / 'test.csv'
TEST_SERIES_CSV = COMP_DIR / 'test_series.csv'
TEST_SERIES_DIR = COMP_DIR / 'test_series'
SAMPLE_SUB = COMP_DIR / 'sample_submission.csv'

import torch

def _pick_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        m = torch.nn.Conv2d(3, 8, 3, padding=1).cuda()
        _ = m(torch.zeros(1, 3, 32, 32, device='cuda')).sum().item()
        return 'cuda'
    except Exception as e:
        print('CUDA inutilisable, repli CPU :', str(e)[:120])
        return 'cpu'

DEVICE = _pick_device()
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

AMP_DTYPE = (torch.bfloat16
             if (DEVICE == 'cuda' and torch.cuda.is_bf16_supported())
             else torch.float16)

# Ordre : ViT-k320, v20h distil, CNX-r7.
# CNX baseline 0.60. ViT et distil se partagent 0.40 (porte 0.015, softmax T=0.05).
N_MODELS = 3
MIX = [
    (0.00, 0.40, 0.60),  # ACL
    (0.00, 0.40, 0.60),  # MCL
    (0.00, 0.40, 0.60),  # Medial Meniscus
    (0.00, 0.40, 0.60),  # Lateral Meniscus
    (0.00, 0.40, 0.60),  # Medial OA
    (0.00, 0.40, 0.60),  # Lateral OA
    (0.00, 0.40, 0.60),  # PF OA
    (0.20, 0.20, 0.60),  # Effusion
    (0.22, 0.18, 0.60),  # Synovitis
    (0.00, 0.40, 0.60),  # Baker's
    (0.17, 0.23, 0.60),  # Contusion
    (0.23, 0.17, 0.60),  # Fracture
]

print('competition dir :', COMP_DIR)
print('poids vit       :', WEIGHTS_VIT, '| existe :', WEIGHTS_VIT.exists())
print('poids v20h dino :', WEIGHTS_V20H, '| existe :', WEIGHTS_V20H.exists())
print('poids cnx       :', WEIGHTS_CNX, '| existe :', WEIGHTS_CNX.exists())
print('device          :', DEVICE)
print('TTA             : cnx vit distil x4 rotation contraste luminosite, 2 GPU')


In [ ]:
# --- Pretraitement DICOM -> volume (24, 224, 224) uint8 --------------------
# (copie self-contained de src/infer/preprocess_dicom.py)
# crop 130 mm + bande 20-80 % (air / bords), identique a la conversion NPZ
import cv2
import numpy as np
import pydicom

N_SLICES, IMG_SIZE = 24, 224
CROP_MM, CROP_REL, BAND = 130.0, 0.70, (0.20, 0.80)


def _slice_sort_key(ds):
    inst = getattr(ds, 'InstanceNumber', None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, 'ImagePositionPatient', None)
    iop = getattr(ds, 'ImageOrientationPatient', None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    return float(ipp[2]) if ipp is not None else 0.0


def _spacing_mm(ds):
    ps = getattr(ds, 'PixelSpacing', None) or getattr(ds, 'ImagerPixelSpacing', None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img, row_mm, col_mm):
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = min(h, max(8, int(round(CROP_MM / row_mm))))
        cw = min(w, max(8, int(round(CROP_MM / col_mm))))
    else:
        ch = min(h, max(8, int(round(h * CROP_REL))))
        cw = min(w, max(8, int(round(w * CROP_REL))))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw]


_SERIES_CACHE = {}

def load_series_dicom(series_dir, img_size=224):
    key = str(Path(series_dir))
    if key not in _SERIES_CACHE:
        _SERIES_CACHE[key] = _series_band(series_dir)
    band = _SERIES_CACHE[key]
    size = int(img_size)
    out = np.empty((band.shape[0], size, size), dtype=np.uint8)
    for i in range(band.shape[0]):
        sl = band[i]
        if sl.shape != (size, size):
            sl = cv2.resize(sl, (size, size), interpolation=cv2.INTER_AREA)
        out[i] = np.clip(np.asarray(sl, dtype=np.float32) * 255.0, 0, 255).round().astype(np.uint8)
    return out


def _series_band(series_dir):
    files = sorted(Path(series_dir).glob('*.dcm'))
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f)))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError(f'aucun DICOM lisible dans {series_dir}')
    datasets.sort(key=_slice_sort_key)

    frames, sp = [], None
    for ds in datasets:
        arr = ds.pixel_array.astype(np.float32)
        arr = arr * float(getattr(ds, 'RescaleSlope', 1.0) or 1.0) \
                  + float(getattr(ds, 'RescaleIntercept', 0.0) or 0.0)
        if getattr(ds, 'PhotometricInterpretation', 'MONOCHROME2') == 'MONOCHROME1':
            arr = arr.max() - arr
        if sp is None:
            sp = _spacing_mm(ds)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f'ndim inattendu {arr.ndim}')
    if not frames:
        raise RuntimeError('aucune coupe')

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped = [_center_crop(im, row_mm, col_mm) for im in frames]
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)
    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    band = vol[a:b + 1]
    idx = np.linspace(0, band.shape[0] - 1, N_SLICES).round().astype(int)
    band = band[idx]
    return band


PROTOCOL_SLOTS = [
    ('Sagittal', 1),
    ('Coronal', 1),
    ('Axial', 1),
    ('Sagittal', 0),
    ('Coronal', 0),
    ('Axial', 0),
]


def _pick_protocol_row(series_rows, plane, fluid):
    '''Un contraste exact (FS ou T1) sur un plan. Pas de repli.'''
    rows = series_rows[series_rows.Anatomical_Plane == plane]
    if len(rows) == 0:
        return None
    if 'Fluid_Sensitive' in rows.columns:
        rows = rows[rows.Fluid_Sensitive == int(fluid)]
    elif fluid == 1 and 'Fat_Suppression' in rows.columns:
        rows = rows[rows.Fat_Suppression == 1]
    elif fluid == 0 and 'Fat_Suppression' in rows.columns:
        rows = rows[rows.Fat_Suppression == 0]
    if len(rows) == 0:
        return None
    return rows.sort_values('SeriesInstanceUID').iloc[0]


LAT_MIN_OFFSET_MM = 20.0


def _lat_side_tag(v):
    if v is None:
        return None
    s = str(v).strip().upper()
    return s[0] if s and s[0] in ('L', 'R') else None


def _lat_center_x(ipp, iop, ps, rows, cols):
    try:
        ipp = np.asarray(list(ipp), dtype=np.float64)
        iop = np.asarray(list(iop), dtype=np.float64)
        ps = np.asarray(list(ps), dtype=np.float64)
        rows, cols = float(rows), float(cols)
    except (TypeError, ValueError):
        return None
    if ipp.size < 3 or iop.size < 6 or ps.size < 2:
        return None
    c = ipp[:3] + iop[:3] * ps[1] * cols / 2.0 + iop[3:6] * ps[0] * rows / 2.0
    x = float(c[0])
    return x if np.isfinite(x) else None


def laterality_from_study_dir(study_dir):
    '''Canon L : tag Laterality, sinon X centre (zone morte 20 mm).'''
    study_dir = Path(study_dir)
    xs, tags = [], []
    if not study_dir.is_dir():
        return None
    for series in sorted(p for p in study_dir.iterdir() if p.is_dir()):
        files = sorted(series.glob('*.dcm'))
        if not files:
            continue
        try:
            ds = pydicom.dcmread(str(files[len(files) // 2]),
                                 stop_before_pixels=True, force=True)
        except Exception:
            continue
        t = _lat_side_tag(getattr(ds, 'Laterality', None))
        if t:
            tags.append(t)
        x = _lat_center_x(
            getattr(ds, 'ImagePositionPatient', None),
            getattr(ds, 'ImageOrientationPatient', None),
            getattr(ds, 'PixelSpacing', None),
            getattr(ds, 'Rows', None), getattr(ds, 'Columns', None))
        if x is not None:
            xs.append(x)
    if tags:
        return tags[0]
    if not xs:
        return None
    m = float(np.median(xs))
    if abs(m) < LAT_MIN_OFFSET_MM:
        return None
    return 'R' if m < 0 else 'L'


def normalise_laterality(vol, plane, lat):
    if lat != 'R':
        return vol
    plane_c = str(plane or '').strip().capitalize()
    if plane_c in ('Coronal', 'Axial'):
        return np.ascontiguousarray(np.flip(vol, axis=-1))
    if plane_c == 'Sagittal':
        return np.ascontiguousarray(np.flip(vol, axis=0))
    return vol


def study_to_tensor(study_dir, series_rows, n_series=3, laterality=False, img_size=224):
    '''(n_series, 24, img_size, img_size) ; n_series=6 : 3 FS + 3 T1, sans repli.'''
    planes = ['Sagittal', 'Coronal', 'Axial']
    size = int(img_size)
    out = np.zeros((n_series, N_SLICES, size, size), dtype=np.float32)
    used = set()
    lat = laterality_from_study_dir(study_dir) if laterality else None

    def _fill(i, pick, plane):
        if pick is None:
            return
        sdir = Path(study_dir) / pick.SeriesInstanceUID
        if not sdir.exists():
            return
        try:
            vol = load_series_dicom(sdir, img_size=size).astype(np.float32) / 255.0
            if laterality:
                vol = normalise_laterality(vol, plane, lat)
            out[i] = vol
            used.add(str(pick.SeriesInstanceUID))
        except Exception as e:
            print(f'  [warn] {pick.SeriesInstanceUID}: {e}')

    if n_series == 6:
        for i, (plane, fluid) in enumerate(PROTOCOL_SLOTS):
            _fill(i, _pick_protocol_row(series_rows, plane, fluid), plane)
        return out

    for i, plane in enumerate(planes):
        rows = series_rows[series_rows.Anatomical_Plane == plane]
        if len(rows) == 0:
            continue
        fluid = rows[rows.Fluid_Sensitive == 1] if 'Fluid_Sensitive' in rows else rows.iloc[0:0]
        if len(fluid) == 0 and 'Fat_Suppression' in rows:
            fluid = rows[rows.Fat_Suppression == 1]
        pick = (fluid if len(fluid) else rows).sort_values('SeriesInstanceUID').iloc[0]
        _fill(i, pick, plane)
    if n_series >= 4:
        t1 = None
        for plane in planes:
            rows = series_rows[series_rows.Anatomical_Plane == plane]
            if 'Fluid_Sensitive' in rows:
                rows = rows[rows.Fluid_Sensitive == 0]
            rows = rows[~rows.SeriesInstanceUID.isin(used)]
            if len(rows):
                t1 = rows.sort_values('SeriesInstanceUID').iloc[0]
                break
        pl = None if t1 is None else str(t1.Anatomical_Plane)
        _fill(3, t1, pl)
    return out


In [ ]:
# --- KneeNet v20h DINO ---
# -*- coding: utf-8 -*-
"""KneeNet v20h — v20b + tete de decision par cible (v20d, 6 slots).

Base v20b inchangee : ResNet-50 RadImageNet, stem A+B, CAM 14x14 dilated,
BiLSTM residuel init 0, BN gelee, 6 slots protocole (FS / T1 par plan).

Levier : v20b partage Linear(C->512) et Linear(512->1) sur les 12 cibles.
Ici, comme v20d (qui n'existait qu'en 3 plans) :

1. `label_embed` (T, C) init 0, ajoute a pooled avant LayerNorm
2. Linear(512 -> T) en **diagonale** : logit_t = W_t · h_t + b_t

A l'init, les T lignes de W (et les T biais) sont copies de la 1re :
epoch 0 = v20b. Les cibles peuvent ensuite diverger.

Entree : (B, 6, 24, H, W) grayscale [0, 1]. Ordre des slots = v20b.
Juger vs subset_v20b_crop130_frac025 (val ET gold, Δ > 0.01), surtout
MCL / Effusion / Medial Meniscus.
"""
from __future__ import annotations

from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
from torchvision.models import resnet50

N_TARGETS = 12
DEFAULT_WEIGHTS = "weights/ResNet50.pt"
MAP_HW = 14
LSTM_HIDDEN = 256
N_PROTOCOL_SLOTS = 6


def _dilate_layer4(layer4: nn.Sequential, dilate: int = 2) -> None:
    """stride 32 -> 16, in-place. Poids 3x3 / 1x1 inchanges."""
    for i, block in enumerate(layer4):
        if i == 0:
            block.conv2.stride = (1, 1)
            if block.downsample is not None:
                block.downsample[0].stride = (1, 1)
        block.conv2.dilation = (dilate, dilate)
        block.conv2.padding = (dilate, dilate)


class _ResNet50Encoder(nn.Module):
    """ResNet-50 jusqu'a layer4 dilated. Cles `backbone.N.*` = v14/v16/v17."""

    def __init__(self) -> None:
        super().__init__()
        try:
            base = resnet50(weights=None)
        except TypeError:
            base = resnet50(pretrained=False)
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        _dilate_layer4(self.backbone[7])
        self.num_features = 2048
        for m in self.modules():
            if isinstance(m, nn.ReLU):
                m.inplace = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


class KneeNetV20h(nn.Module):
    arch_id = "kneenet_v20h"

    def __init__(self, backbone: str = "resnet50", n_series: int = N_PROTOCOL_SLOTS,
                 n_targets: int = N_TARGETS, head_dim: int = 512,
                 dropout: float = 0.2, pretrained: bool = True,
                 encode_chunk: int = 16, max_depth: int = 64,
                 backbone_weights: str = DEFAULT_WEIGHTS,
                 lstm_hidden: int = LSTM_HIDDEN) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        self.backbone_name = str(backbone)
        self.map_hw = MAP_HW
        self.lstm_hidden = int(lstm_hidden)

        self.encoder = _ResNet50Encoder()
        if pretrained:
            self._load_radimagenet(backbone_weights)
        feat_dim = self.encoder.num_features

        self.plane_embed = nn.Parameter(torch.zeros(n_series, feat_dim))
        self.slice_embed = nn.Parameter(torch.zeros(max_depth, feat_dim))
        self.attn_score = nn.Linear(feat_dim, n_targets)
        self.spatial_score = nn.Conv2d(feat_dim, n_targets, kernel_size=1)
        nn.init.zeros_(self.spatial_score.weight)
        nn.init.zeros_(self.spatial_score.bias)

        self.slice_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=self.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_proj = nn.Linear(2 * self.lstm_hidden, feat_dim)
        nn.init.zeros_(self.lstm_proj.weight)
        nn.init.zeros_(self.lstm_proj.bias)

        self.label_embed = nn.Parameter(torch.zeros(n_targets, feat_dim))
        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.head_out = nn.Linear(head_dim, n_targets)
        with torch.no_grad():
            w0 = self.head_out.weight[0].detach().clone()
            b0 = self.head_out.bias[0].detach().clone()
            self.head_out.weight.copy_(w0.unsqueeze(0).expand_as(self.head_out.weight))
            self.head_out.bias.copy_(b0.expand_as(self.head_out.bias))

        self.stem_mix = nn.Conv2d(3, 3, kernel_size=1, bias=False)
        self._init_stem_mix()
        self.register_buffer(
            "img_mean", torch.zeros(1, 3, 1, 1), persistent=True,
        )
        self.register_buffer(
            "img_std", torch.ones(1, 3, 1, 1), persistent=True,
        )

    def extra_log(self) -> dict[str, float]:
        """Divergence des lignes de head_out vs la 1re (init liee = 0)."""
        w = self.head_out.weight.detach()
        b = self.head_out.bias.detach()
        dw = (w - w[0]).abs().mean()
        db = (b - b[0]).abs().mean()
        return {
            "head_out_w_div": float(dw),
            "head_out_b_div": float(db),
            "label_embed_abs": float(self.label_embed.detach().abs().mean()),
        }

    def _init_stem_mix(self) -> None:
        """out_c = in_G pour les 3 canaux -> R=G=B=k au depart."""
        with torch.no_grad():
            self.stem_mix.weight.zero_()
            self.stem_mix.weight[:, 1, 0, 0] = 1.0

    def _load_radimagenet(self, path: str) -> None:
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(
                f"poids RadImageNet introuvables : {p.resolve()}"
            )
        sd = torch.load(p, map_location="cpu", weights_only=False)
        self.encoder.load_state_dict(sd, strict=True)
        print(
            f"RadImageNet ResNet-50 charge : {p} | CAM {MAP_HW}x{MAP_HW} "
            f"| BiLSTM hidden {self.lstm_hidden} | slots {self.n_series} "
            f"| tete par cible (diag, init liee)",
            flush=True,
        )

    def train(self, mode: bool = True):
        super().train(mode)
        for m in self.encoder.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def encoder_parameters(self):
        return self.encoder.parameters()

    def head_parameters(self):
        for name, p in self.named_parameters():
            if not name.startswith("encoder."):
                yield p

    def _encode_chunks(self, rgb: torch.Tensor) -> torch.Tensor:
        """rgb : (N, 3, H, W) dans [0, 1] -> (N, C, 14, 14)."""
        n = rgb.size(0)
        c = self.encoder.num_features
        if n == 0:
            return rgb.new_zeros(0, c, MAP_HW, MAP_HW)
        outs = []
        chunk = max(1, self.encode_chunk)
        for i in range(0, n, chunk):
            part = rgb[i:i + chunk]
            if self.training:
                feats = checkpoint(self.encoder, part, use_reentrant=False)
            else:
                feats = self.encoder(part)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _spatial_pool(self, maps: torch.Tensor, return_attn: bool = False):
        """maps (N, C, H, W) -> gap (N, C), local (N, T, C) [, attn (N, T, H, W)]."""
        n, c, h, w = maps.shape
        gap = maps.mean(dim=(-2, -1))
        logits = self.spatial_score(maps)
        attn = torch.softmax(
            logits.float().reshape(n, self.n_targets, h * w), dim=-1,
        ).to(maps.dtype)
        flat = maps.reshape(n, c, h * w)
        local = torch.einsum("nth,nch->ntc", attn, flat)
        if return_attn:
            return gap, local, attn.reshape(n, self.n_targets, h, w)
        return gap, local

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """Triplet A+B pour les tokens valides. x (B,S,D,H,W), idx plat."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        center = x[bs, ss, dd]
        rgb = torch.stack(
            [
                0.5 + 0.5 * (center - x[bs, ss, d_m]),
                center,
                0.5 + 0.5 * (x[bs, ss, d_p] - center),
            ],
            dim=1,
        )
        rgb = self.stem_mix(rgb)
        return (rgb - self.img_mean) / self.img_std

    def _temporal_delta(self, gaps: torch.Tensor, b: int, s: int,
                        d: int) -> torch.Tensor:
        """gaps (BSD, C) -> residu LSTM (B, S, D, C). Epoch 0 : 0."""
        c = gaps.size(-1)
        seq = gaps.reshape(b * s, d, c)
        with torch.autocast("cuda", enabled=False):
            h, _ = self.slice_lstm(seq.float())
            delta = self.lstm_proj(h)
        return delta.to(dtype=gaps.dtype).reshape(b, s, d, c)

    def _head_logits(self, pooled: torch.Tensor) -> torch.Tensor:
        """pooled (B, T, C) -> logits (B, T). Epoch 0 = tete v20b partagee."""
        pooled = pooled + self.label_embed.to(dtype=pooled.dtype)
        hidden = self.head(self.norm(pooled))
        return self.head_out(hidden).diagonal(dim1=-2, dim2=-1)

    def _slice_pool(self, x: torch.Tensor, valid: torch.Tensor,
                    gaps: torch.Tensor, local: torch.Tensor):
        """gaps (BSD,C), local (BSD,T,C) -> logits (B,T), slice_attn (B,S,D,T)."""
        b, s, d = valid.shape
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, -1)
        feats = gaps.reshape(b, s, d, -1) + delta + pe + se
        scores = self.attn_score(feats)
        scores = scores.masked_fill(~valid.unsqueeze(-1), -1.0e4)
        scores = scores.reshape(b, s * d, self.n_targets)
        attn = torch.softmax(scores.float(), dim=1).to(feats.dtype)
        tokens = local.reshape(b, s, d, self.n_targets, -1)
        tokens = tokens + delta.unsqueeze(3) + pe.unsqueeze(3) + se.unsqueeze(3)
        tokens = tokens.reshape(b, s * d, self.n_targets, -1)
        pooled = torch.einsum("bnt,bntc->btc", attn, tokens)
        logits = self._head_logits(pooled)
        return logits, attn.reshape(b, s, d, self.n_targets)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, S, D, H, W) dans [0, 1] -> logits (B, 12)."""
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        if d > self.slice_embed.size(0):
            raise ValueError(
                f"D={d} > max_depth={self.slice_embed.size(0)} : "
                "augmenter max_depth"
            )

        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            if gaps.dtype != maps.dtype:
                gaps = gaps.to(dtype=maps.dtype)
                local = local.to(dtype=maps.dtype)
            gap, loc = self._spatial_pool(maps)
            gaps[idx] = gap
            local[idx] = loc
        logits, _attn = self._slice_pool(x, valid, gaps, local)
        return logits

    @torch.no_grad()
    def interpret(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        """Forward eval + cartes. spatial_attn : (B, S, D, T, 14, 14)."""
        self.eval()
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        hw = MAP_HW
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        spatial = x.new_zeros(n_tok, self.n_targets, hw, hw)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            if gaps.dtype != maps.dtype:
                gaps = gaps.to(dtype=maps.dtype)
                local = local.to(dtype=maps.dtype)
                spatial = spatial.to(dtype=maps.dtype)
            gap, loc, spat = self._spatial_pool(maps, return_attn=True)
            gaps[idx] = gap
            local[idx] = loc
            spatial[idx] = spat
        logits, slice_attn = self._slice_pool(x, valid, gaps, local)
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, 1, -1)
        cam_tokens = (
            local.reshape(b, s, d, self.n_targets, -1)
            + delta.unsqueeze(3) + pe + se
        )
        return {
            "logits": logits,
            "slice_attn": slice_attn,
            "spatial_attn": spatial.reshape(
                b, s, d, self.n_targets, hw, hw,
            ),
            "cam_tokens": cam_tokens,
            "valid": valid,
        }


In [ ]:
# --- KneeNet ViT-e, poids ViT-k320 ---
# -*- coding: utf-8 -*-
"""KneeNet-ViT-e — v1 (CAM+BiLSTM) + stem RGB k-1/k/k+1 de d.

Famille CLS (c/d) : le 2.5D aide (contusion 0.66 -> 0.72) mais le gold
reste ~0.74 vs v1 0.80. Le plafond CLS+v8 est sous v1. On recolle le
graphe qui score (CAM patches + LSTM) et on y met le seul levier D
qui a marche.

  - stem : R=k-1, G=k, B=k+1, stem_mix init G (epoch 0 == v1)
  - cartes patch 14x14, CAM + GAP, BiLSTM init 0 (comme v1)
  - LayerScale gele (recette c/d)

Recette : freeze 1, lr_backbone 3e-5 (v1 tenait), re-warmup 1 epoch.
Option `drop_path_rate` (YAML) : stochastic depth timm, OFF pendant
le freeze (epoch 1 = identity e). Defaut 0 = graphe e inchange.
Poids : `weights/dinov3-vitb16-lvd1689m/model.safetensors`
Entree : (B, 6, 24, H, W) grayscale [0, 1]. Slots = v20b.
"""
from __future__ import annotations

from pathlib import Path

import timm
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

N_TARGETS = 12
DEFAULT_WEIGHTS = "weights/dinov3-vitb16-lvd1689m"
LSTM_HIDDEN = 256
N_PROTOCOL_SLOTS = 6
PATCH_SIZE = 16
N_PREFIX = 5
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
TIMM_NAME = "vit_base_patch16_dinov3.lvd1689m"


def _weights_file(path: str | Path) -> Path:
    p = Path(path)
    if p.is_dir():
        for name in ("model.safetensors", "pytorch_model.bin"):
            cand = p / name
            if cand.is_file():
                return cand
        raise FileNotFoundError(
            f"aucun checkpoint dans {p.resolve()} "
            "(attendu model.safetensors)"
        )
    if not p.is_file():
        raise FileNotFoundError(f"poids DINOv3 introuvables : {p.resolve()}")
    return p


class _Dinov3VitEncoder(nn.Module):
    """ViT-B/16 DINOv3 -> cartes (N, 768, H/16, W/16)."""

    def __init__(self, weights_path: str = DEFAULT_WEIGHTS,
                 pretrained: bool = True,
                 drop_path_rate: float = 0.0,
                 grad_checkpoint: bool = True) -> None:
        super().__init__()
        kwargs = dict(
            num_classes=0,
            global_pool="",
            dynamic_img_size=True,
        )
        dp = float(drop_path_rate or 0.0)
        if dp > 0:
            kwargs["drop_path_rate"] = dp
        if pretrained:
            ckpt = _weights_file(weights_path)
            if ckpt.suffix == ".pt":
                self.backbone = timm.create_model(
                    TIMM_NAME, pretrained=False, **kwargs,
                )
                sd = torch.load(ckpt, map_location="cpu", weights_only=False)
                self.load_state_dict(sd, strict=True)
                print(
                    f"DINOv3 ViT-B/16 charge : {ckpt} | {len(sd)} tenseurs",
                    flush=True,
                )
            else:
                self.backbone = timm.create_model(
                    TIMM_NAME,
                    pretrained=True,
                    pretrained_cfg_overlay={"file": str(ckpt)},
                    **kwargs,
                )
                print(f"DINOv3 ViT-B/16 charge : {ckpt}", flush=True)
        else:
            self.backbone = timm.create_model(
                TIMM_NAME, pretrained=False, **kwargs,
            )
        self.backbone.set_grad_checkpointing(bool(grad_checkpoint))
        self.num_features = int(self.backbone.num_features)
        self.patch_size = PATCH_SIZE
        self.n_prefix = int(getattr(
            self.backbone, "num_prefix_tokens", N_PREFIX))
        self.drop_path_rate = dp
        self._drop_path: list[tuple[nn.Module, float]] = []
        for m in self.backbone.modules():
            if m.__class__.__name__ == "DropPath":
                self._drop_path.append((m, float(m.drop_prob)))
        if dp > 0:
            print(
                f"drop_path {dp:g} | {len(self._drop_path)} modules "
                f"(off pendant freeze)",
                flush=True,
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x (N, 3, H, W) -> (N, C, h, w) patch tokens."""
        tokens = self.backbone.forward_features(x)
        patches = tokens[:, self.n_prefix:]
        n, n_hw, c = patches.shape
        side = int(n_hw ** 0.5)
        if side * side != n_hw:
            raise ValueError(
                f"grille patch non carree : {n_hw} tokens "
                f"(H={x.shape[-2]}, W={x.shape[-1]})"
            )
        return patches.transpose(1, 2).reshape(n, c, side, side)


class KneeNetVitE(nn.Module):
    arch_id = "kneenet_vit_e"

    def __init__(self, backbone: str = TIMM_NAME,
                 n_series: int = N_PROTOCOL_SLOTS,
                 n_targets: int = N_TARGETS, head_dim: int = 512,
                 dropout: float = 0.2, pretrained: bool = True,
                 encode_chunk: int = 4, max_depth: int = 64,
                 backbone_weights: str = DEFAULT_WEIGHTS,
                 lstm_hidden: int = LSTM_HIDDEN,
                 drop_path_rate: float = 0.0,
                 grad_checkpoint: bool = True) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        # False sur grosse carte : sinon le ViT est decoupe en petits
        # lancements et le GPU attend Python entre deux blocs.
        self.grad_checkpoint = bool(grad_checkpoint)
        self.backbone_name = str(backbone)
        self.patch_size = PATCH_SIZE
        self.lstm_hidden = int(lstm_hidden)
        self.drop_path_rate = float(drop_path_rate or 0.0)

        self.encoder = _Dinov3VitEncoder(
            weights_path=backbone_weights, pretrained=pretrained,
            drop_path_rate=self.drop_path_rate,
            grad_checkpoint=self.grad_checkpoint,
        )
        self._freeze_layer_scale()
        feat_dim = self.encoder.num_features

        self.stem_mix = nn.Conv2d(3, 3, kernel_size=1, bias=False)
        self._init_stem_mix()

        self.plane_embed = nn.Parameter(torch.zeros(n_series, feat_dim))
        self.slice_embed = nn.Parameter(torch.zeros(max_depth, feat_dim))
        self.attn_score = nn.Linear(feat_dim, n_targets)
        self.spatial_score = nn.Conv2d(feat_dim, n_targets, kernel_size=1)
        nn.init.zeros_(self.spatial_score.weight)
        nn.init.zeros_(self.spatial_score.bias)

        self.slice_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=self.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_proj = nn.Linear(2 * self.lstm_hidden, feat_dim)
        nn.init.zeros_(self.lstm_proj.weight)
        nn.init.zeros_(self.lstm_proj.bias)

        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_dim, 1),
        )
        self.register_buffer(
            "img_mean",
            torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1),
            persistent=True,
        )
        self.register_buffer(
            "img_std",
            torch.tensor(IMAGENET_STD).view(1, 3, 1, 1),
            persistent=True,
        )
        dp_txt = (f" | drop_path {self.drop_path_rate:g}"
                  if self.drop_path_rate > 0 else "")
        print(
            f"KneeNet-ViT-e | CAM patch {PATCH_SIZE} | BiLSTM "
            f"| RGB k-1/k/k+1{dp_txt} | dim {feat_dim} | slots {self.n_series}",
            flush=True,
        )

    def _init_stem_mix(self) -> None:
        """out = in_G -> epoch 0 identique a v1."""
        with torch.no_grad():
            self.stem_mix.weight.zero_()
            self.stem_mix.weight[:, 1, 0, 0] = 1.0

    @staticmethod
    def _is_layerscale(name: str) -> bool:
        return name.rsplit(".", 1)[-1].startswith("gamma")

    def _freeze_layer_scale(self) -> None:
        n = 0
        for name, p in self.encoder.named_parameters():
            if self._is_layerscale(name):
                p.requires_grad = False
                n += 1
        print(f"LayerScale gele : {n} tenseurs gamma", flush=True)

    def set_encoder_frozen(self, flag: bool) -> None:
        for name, p in self.encoder.named_parameters():
            p.requires_grad = (not flag) and not self._is_layerscale(name)
        self.stem_mix.weight.requires_grad = not flag
        # DropPath stochastique seulement une fois l'encodeur degele,
        # sinon l'epoch 1 n'est plus identity-e.
        for m, p0 in self.encoder._drop_path:
            m.drop_prob = 0.0 if flag else p0

    def encoder_parameters(self):
        for p in self.encoder.parameters():
            if p.requires_grad:
                yield p
        if self.stem_mix.weight.requires_grad:
            yield self.stem_mix.weight

    def head_parameters(self):
        for name, p in self.named_parameters():
            if name.startswith("encoder.") or name.startswith("stem_mix."):
                continue
            yield p

    def _encode_chunks(self, rgb: torch.Tensor) -> torch.Tensor:
        n = rgb.size(0)
        c = self.encoder.num_features
        if n == 0:
            h = rgb.size(-2) // self.patch_size
            w = rgb.size(-1) // self.patch_size
            return rgb.new_zeros(0, c, h, w)
        outs = []
        chunk = max(1, self.encode_chunk)
        for i in range(0, n, chunk):
            part = rgb[i:i + chunk]
            if self.training and self.grad_checkpoint:
                feats = checkpoint(self.encoder, part, use_reentrant=False)
            else:
                feats = self.encoder(part)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _spatial_pool(self, maps: torch.Tensor, return_attn: bool = False):
        n, c, h, w = maps.shape
        gap = maps.mean(dim=(-2, -1))
        logits = self.spatial_score(maps)
        attn = torch.softmax(
            logits.float().reshape(n, self.n_targets, h * w), dim=-1,
        ).to(maps.dtype)
        flat = maps.reshape(n, c, h * w)
        local = torch.einsum("nth,nch->ntc", attn, flat)
        if return_attn:
            return gap, local, attn.reshape(n, self.n_targets, h, w)
        return gap, local

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """R=k-1, G=k, B=k+1 puis stem_mix. x (B,S,D,H,W), idx plat."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        rgb = torch.stack(
            [x[bs, ss, d_m], x[bs, ss, dd], x[bs, ss, d_p]],
            dim=1,
        )
        rgb = self.stem_mix(rgb)
        return (rgb - self.img_mean) / self.img_std

    def _temporal_delta(self, gaps: torch.Tensor, b: int, s: int,
                        d: int) -> torch.Tensor:
        c = gaps.size(-1)
        seq = gaps.reshape(b * s, d, c)
        with torch.autocast("cuda", enabled=False):
            h, _ = self.slice_lstm(seq.float())
            delta = self.lstm_proj(h)
        return delta.to(dtype=gaps.dtype).reshape(b, s, d, c)

    def _slice_pool(self, x: torch.Tensor, valid: torch.Tensor,
                    gaps: torch.Tensor, local: torch.Tensor):
        b, s, d = valid.shape
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, -1)
        feats = gaps.reshape(b, s, d, -1) + delta + pe + se
        scores = self.attn_score(feats)
        scores = scores.masked_fill(~valid.unsqueeze(-1), -1.0e4)
        scores = scores.reshape(b, s * d, self.n_targets)
        attn = torch.softmax(scores.float(), dim=1).to(feats.dtype)
        tokens = local.reshape(b, s, d, self.n_targets, -1)
        tokens = tokens + delta.unsqueeze(3) + pe.unsqueeze(3) + se.unsqueeze(3)
        tokens = tokens.reshape(b, s * d, self.n_targets, -1)
        pooled = torch.einsum("bnt,bntc->btc", attn, tokens)
        logits = self.head(self.norm(pooled)).squeeze(-1)
        return logits, attn.reshape(b, s, d, self.n_targets)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        if d > self.slice_embed.size(0):
            raise ValueError(
                f"D={d} > max_depth={self.slice_embed.size(0)} : "
                "augmenter max_depth"
            )
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            gap, loc = self._spatial_pool(maps)
            if gaps.dtype != gap.dtype:
                gaps = gaps.to(dtype=gap.dtype)
            if local.dtype != loc.dtype:
                local = local.to(dtype=loc.dtype)
            gaps[idx] = gap
            local[idx] = loc
        logits, _attn = self._slice_pool(x, valid, gaps, local)
        return logits

    @torch.no_grad()
    def interpret(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        self.eval()
        b, s, d, img_h, img_w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        hw_h, hw_w = img_h // self.patch_size, img_w // self.patch_size
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        spatial = x.new_zeros(n_tok, self.n_targets, hw_h, hw_w)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            gap, loc, spat = self._spatial_pool(maps, return_attn=True)
            if gaps.dtype != gap.dtype:
                gaps = gaps.to(dtype=gap.dtype)
            if local.dtype != loc.dtype:
                local = local.to(dtype=loc.dtype)
            if spatial.dtype != spat.dtype:
                spatial = spatial.to(dtype=spat.dtype)
            gaps[idx] = gap
            local[idx] = loc
            spatial[idx] = spat
        logits, slice_attn = self._slice_pool(x, valid, gaps, local)
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, 1, -1)
        cam_tokens = (
            local.reshape(b, s, d, self.n_targets, -1)
            + delta.unsqueeze(3) + pe + se
        )
        return {
            "logits": logits,
            "slice_attn": slice_attn,
            "spatial_attn": spatial.reshape(
                b, s, d, self.n_targets, hw_h, hw_w,
            ),
            "cam_tokens": cam_tokens,
            "valid": valid,
        }


In [ ]:
# --- KneeNet CNX-r7, namespace isole ---
import json as _json
_cnx_src = _json.loads("\"# -*- coding: utf-8 -*-\\n\\\"\\\"\\\"KneeNet-CNX-r7 \\u2014 tete R7 (Heliosli) sur l'entree hybrid025.\\n\\nMeme tenseur que CNX-a : (B, 6, 24, 224, 224) grayscale [0, 1], slots\\nv20b. Le cache px384_v4 n'est pas reconstruit : c'est leur rendu\\nprecalcule (384 px, vues 100 mm, meta DICOM). Ici chaque coupe valide\\nest une fenetre 2.5D (k-1, k, k+1), comme les 3 canaux de R7.\\n\\nGraphe, repris de compact_train.Net (spatial + extent + syn_decomp) :\\n  - ConvNeXt-Small stride 32, drop_path 0.1, carte brute (sans head.norm).\\n  - Attention spatiale par pathologie (requete nulle a l'init = moyenne).\\n  - LayerNorm de tete apres le pooling, par classe.\\n  - Attention par pathologie sur les coupes du study.\\n  - Etendue OA : gamma * log1p(aire/100), gamma nul a l'init.\\n    Aire = produit sagittal x coronal (OA TF) ou axial x sagittal (PF).\\n    Epaisseur de dalle : 3.5 mm x nombre de coupes valides du slot\\n    (le zip crop130 n'a pas le spacing DICOM).\\n  - Synovite = part specifique + beta * stopgrad(logit epanchement).\\n\\nPas de tetes d'etat ni d'auxiliaire fastMRI : les cibles sont v3b (12\\ncolonnes). Ces tetes ne changent pas les logits tant que leur porte\\nvaut 0, et il n'y a pas de labels d'etat pour les apprendre.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport math\\nfrom pathlib import Path\\n\\nimport timm\\nprint('timm', timm.__version__, flush=True)\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\nfrom torch.utils.checkpoint import checkpoint\\n\\nN_TARGETS = 12\\nTIMM_NAME = \\\"convnext_small.fb_in22k_ft_in1k\\\"\\nDEFAULT_WEIGHTS = \\\"weights/convnext_small-fb_in22k_ft_in1k\\\"\\nN_PROTOCOL_SLOTS = 6\\nHIDDEN = 384\\nOA_MED, OA_LAT, OA_PF = 4, 5, 6\\nSYN, EFF = 8, 7\\n# slots v20b : SAG_FS, COR_FS, AX_FS, SAG_T1, COR_T1, AX_T1\\nSLOT_PLANE = (0, 1, 2, 0, 1, 2)\\nSLOT_FLUID = (1, 1, 1, 0, 0, 0)\\nSLAB_MM = 3.5\\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\\nIMAGENET_STD = (0.229, 0.224, 0.225)\\n\\n\\ndef _weights_file(path: str | Path) -> Path:\\n    p = Path(path)\\n    if p.is_dir():\\n        for name in (\\\"model.safetensors\\\", \\\"pytorch_model.bin\\\"):\\n            cand = p / name\\n            if cand.is_file():\\n                return cand\\n        raise FileNotFoundError(\\n            f\\\"aucun checkpoint dans {p.resolve()} \\\"\\n            \\\"(attendu model.safetensors)\\\"\\n        )\\n    if not p.is_file():\\n        raise FileNotFoundError(f\\\"poids ConvNeXt introuvables : {p.resolve()}\\\")\\n    return p\\n\\n\\nclass _ConvNextEncoder(nn.Module):\\n    \\\"\\\"\\\"Carte stride 32 avant head.norm. drop_path 0.1, comme R7.\\\"\\\"\\\"\\n\\n    def __init__(self, name: str = TIMM_NAME,\\n                 weights_path: str = DEFAULT_WEIGHTS,\\n                 pretrained: bool = True) -> None:\\n        super().__init__()\\n        kwargs = dict(\\n            num_classes=0, global_pool=\\\"\\\", output_stride=32,\\n            drop_path_rate=0.1,\\n        )\\n        if pretrained:\\n            ckpt = _weights_file(weights_path)\\n            self.backbone = timm.create_model(\\n                name, pretrained=True,\\n                pretrained_cfg_overlay={\\\"file\\\": str(ckpt)}, **kwargs,\\n            )\\n            print(f\\\"ConvNeXt charge : {name} | {ckpt} | stride 32 | drop_path 0.1\\\",\\n                  flush=True)\\n        else:\\n            self.backbone = timm.create_model(\\n                name.split(\\\".\\\", 1)[0], pretrained=False, **kwargs,\\n            )\\n        self.num_features = int(self.backbone.num_features)\\n        cfg = self.backbone.pretrained_cfg if pretrained else {}\\n        self.mean = tuple(float(v) for v in cfg.get(\\\"mean\\\", IMAGENET_MEAN))\\n        self.std = tuple(float(v) for v in cfg.get(\\\"std\\\", IMAGENET_STD))\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        return self.backbone.forward_features(x)\\n\\n\\nclass KneeNet(nn.Module):\\n    arch_id = \\\"kneenet_cnx_r7\\\"\\n\\n    def __init__(self, backbone: str = TIMM_NAME,\\n                 n_series: int = N_PROTOCOL_SLOTS,\\n                 n_targets: int = N_TARGETS, hidden: int = HIDDEN,\\n                 dropout: float = 0.1, pretrained: bool = True,\\n                 encode_chunk: int = 4, max_depth: int = 64,\\n                 backbone_weights: str = DEFAULT_WEIGHTS) -> None:\\n        super().__init__()\\n        self.n_series = n_series\\n        self.n_targets = n_targets\\n        self.encode_chunk = int(encode_chunk)\\n        self.backbone_name = str(backbone)\\n        self.hidden = int(hidden)\\n        self.max_depth = int(max_depth)\\n\\n        self.encoder = _ConvNextEncoder(\\n            name=self.backbone_name, weights_path=backbone_weights,\\n            pretrained=pretrained,\\n        )\\n        d = self.encoder.num_features\\n        norm = self.encoder.backbone.head.norm\\n        if isinstance(norm, nn.Identity):\\n            raise RuntimeError(\\\"ConvNeXt sans head.norm : le pooling R7 en a besoin\\\")\\n        self.sp_head_norm = norm\\n\\n        self.sp_key = nn.Linear(d, 128)\\n        self.sp_pos = nn.Linear(16, 128, bias=False)\\n        nn.init.zeros_(self.sp_pos.weight)\\n        self.sp_q = nn.Parameter(torch.zeros(n_targets, 128))\\n\\n        self.proj = nn.Sequential(nn.Linear(d, self.hidden), nn.LayerNorm(self.hidden))\\n        self.slot = nn.Embedding(10, self.hidden)\\n        self.view = nn.Embedding(6, self.hidden)\\n        self.geom = nn.Linear(4, self.hidden, bias=False)\\n        nn.init.zeros_(self.slot.weight)\\n        nn.init.zeros_(self.view.weight)\\n        nn.init.zeros_(self.geom.weight)\\n        self.drop = nn.Dropout(dropout)\\n        self.win_proj = nn.Linear(self.hidden, self.hidden)\\n        self.q_win = nn.Parameter(torch.randn(n_targets, self.hidden) * 0.02)\\n        self.out_norm = nn.LayerNorm(self.hidden)\\n        self.cls_w = nn.Parameter(torch.randn(n_targets, self.hidden) * 0.02)\\n        self.bias = nn.Parameter(torch.zeros(n_targets))\\n\\n        self.ext_u = nn.Parameter(torch.zeros(n_targets, self.hidden))\\n        self.ext_b = nn.Parameter(torch.full((n_targets,), -4.0))\\n        self.ext_gamma = nn.Parameter(torch.zeros(n_targets))\\n        self.ext_view = nn.Parameter(torch.zeros(n_targets, 6))\\n        em = torch.zeros(n_targets)\\n        em[OA_MED] = em[OA_LAT] = em[OA_PF] = 1.0\\n        self.register_buffer(\\\"ext_mask\\\", em)\\n        self.syn_beta = nn.Parameter(torch.zeros(()))\\n\\n        self.register_buffer(\\n            \\\"img_mean\\\", torch.tensor(self.encoder.mean).view(1, 3, 1, 1),\\n            persistent=True,\\n        )\\n        self.register_buffer(\\n            \\\"img_std\\\", torch.tensor(self.encoder.std).view(1, 3, 1, 1),\\n            persistent=True,\\n        )\\n        self.register_buffer(\\\"slot_plane\\\", torch.tensor(SLOT_PLANE, dtype=torch.long))\\n        self.register_buffer(\\\"slot_fluid\\\", torch.tensor(SLOT_FLUID, dtype=torch.float32))\\n        print(\\n            f\\\"KneeNet-CNX-r7 | {self.backbone_name} | hidden {self.hidden} \\\"\\n            f\\\"| spatial + fenetres + etendue OA + syn_beta \\\"\\n            f\\\"| slots {self.n_series} | dim {d}\\\",\\n            flush=True,\\n        )\\n\\n    def set_encoder_frozen(self, flag: bool) -> None:\\n        for p in self.encoder.parameters():\\n            p.requires_grad = not flag\\n\\n    def encoder_parameters(self):\\n        yield from self.encoder.parameters()\\n\\n    def head_parameters(self):\\n        for name, p in self.named_parameters():\\n            if name.startswith(\\\"encoder.\\\"):\\n                continue\\n            yield p\\n\\n    @torch.no_grad()\\n    def extra_log(self) -> dict[str, float]:\\n        return {\\n            \\\"syn_beta\\\": float(self.syn_beta.detach().cpu()),\\n            \\\"ext_gamma_abs\\\": float(self.ext_gamma.detach().abs().mean()),\\n            \\\"sp_q_abs\\\": float(self.sp_q.detach().abs().mean()),\\n        }\\n\\n    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:\\n        \\\"\\\"\\\"Trois coupes adjacentes -> 3 canaux. x (B,S,D,H,W).\\\"\\\"\\\"\\n        _b, s, d, _h, _w = x.shape\\n        bs = idx // (s * d)\\n        rest = idx % (s * d)\\n        ss = rest // d\\n        dd = rest % d\\n        d_m = (dd - 1).clamp(0, d - 1)\\n        d_p = (dd + 1).clamp(0, d - 1)\\n        rgb = torch.stack(\\n            [x[bs, ss, d_m], x[bs, ss, dd], x[bs, ss, d_p]],\\n            dim=1,\\n        )\\n        return (rgb - self.img_mean) / self.img_std\\n\\n    def _encode_indexed(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:\\n        chunk = max(1, self.encode_chunk)\\n        outs = []\\n        for i in range(0, int(idx.numel()), chunk):\\n            rgb = self._stem_rgb(x, idx[i:i + chunk])\\n            if self.training:\\n                feats = checkpoint(self.encoder, rgb, use_reentrant=False)\\n            else:\\n                feats = self.encoder(rgb)\\n            outs.append(feats)\\n        return torch.cat(outs, dim=0)\\n\\n    def _pos_enc(self, h: int, w: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:\\n        yy, xx = torch.meshgrid(\\n            torch.linspace(-1, 1, h, device=device),\\n            torch.linspace(-1, 1, w, device=device),\\n            indexing=\\\"ij\\\",\\n        )\\n        fr = torch.tensor([1.0, 2.0, 4.0, 8.0], device=device) * (math.pi / 2)\\n        yy = yy.reshape(-1, 1) * fr\\n        xx = xx.reshape(-1, 1) * fr\\n        return torch.cat([yy.sin(), yy.cos(), xx.sin(), xx.cos()], 1).to(dtype)\\n\\n    def spatial_pool(self, maps: torch.Tensor) -> torch.Tensor:\\n        \\\"\\\"\\\"maps (N, d, h, w) -> (N, C, d). Requete nulle = moyenne des positions.\\\"\\\"\\\"\\n        _n, _d, h, w = maps.shape\\n        f = maps.flatten(2).transpose(1, 2)\\n        pe = self._pos_enc(h, w, f.device, f.dtype)\\n        kin = F.layer_norm(f.float(), (f.shape[-1],)).to(f.dtype)\\n        k = torch.tanh(self.sp_key(kin) + self.sp_pos(pe)[None])\\n        att = torch.einsum(\\\"npk,ck->npc\\\", k, self.sp_q.to(k.dtype)).float().softmax(1).to(f.dtype)\\n        g = torch.einsum(\\\"npc,npd->ncd\\\", att, f)\\n        nrm = self.sp_head_norm\\n        g = F.layer_norm(g.float(), (g.shape[-1],), nrm.weight, nrm.bias, nrm.eps).to(f.dtype)\\n        return g\\n\\n    def _window_meta(self, valid: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\\n        \\\"\\\"\\\"valid (B,S,D) -> mask (B,W), meta (B,W,9).\\\"\\\"\\\"\\n        b, s, d = valid.shape\\n        device = valid.device\\n        rel = torch.linspace(0, 1, d, device=device)\\n        n_valid = valid.flatten(2).sum(-1).clamp(min=1).to(torch.float32)\\n        slot = torch.arange(s, device=device)\\n        plane = self.slot_plane[:s].to(device)\\n        fluid = self.slot_fluid[:s].to(device)\\n        meta = valid.new_zeros(b, s, d, 9, dtype=torch.float32)\\n        meta[..., 0] = slot.view(1, s, 1).float()\\n        meta[..., 2] = rel.view(1, 1, d)\\n        meta[..., 3] = math.log1p(SLAB_MM)\\n        meta[..., 4] = 1.0\\n        meta[..., 5] = fluid.view(1, s, 1)\\n        meta[..., 6] = slot.view(1, s, 1).float()\\n        meta[..., 7] = plane.view(1, s, 1).float()\\n        meta[..., 8] = n_valid.view(b, s, 1)\\n        return valid.reshape(b, s * d), meta.reshape(b, s * d, 9)\\n\\n    def extent_area(self, e: torch.Tensor, meta: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\\n        \\\"\\\"\\\"e (B,W,C) -> aire proxy (B,C), mm^2. Meme combinaison de plans que R7.\\\"\\\"\\\"\\n        v = 6\\n        sid = meta[..., 6].long()\\n        s_max = max(16, int(sid.max()) + 1)\\n        vid = meta[..., 1].long().clamp(0, v - 1)\\n        q = torch.sigmoid(e.float()) * mask[..., None]\\n        ohv = F.one_hot(sid * v + vid, s_max * v).float() * mask[..., None]\\n        cntv = ohv.sum(1)\\n        mean_v = (\\n            torch.einsum(\\\"bwk,bwc->bkc\\\", ohv, q) / cntv.clamp_min(1)[..., None]\\n        ).view(e.shape[0], s_max, v, -1)\\n        present = (cntv > 0).view(e.shape[0], s_max, v, 1)\\n        wv = self.ext_view.float().t()[None, None].masked_fill(~present, -1e4).softmax(2) * present\\n        mean = (wv * mean_v).sum(2)\\n        oh = F.one_hot(sid, s_max).float() * mask[..., None]\\n        cnt = oh.sum(1)\\n        span = torch.expm1(meta[..., 3].float()).clamp_min(0) * meta[..., 8].float()\\n        span_s = torch.einsum(\\\"bws,bw->bs\\\", oh, span) / cnt.clamp_min(1)\\n        plane_s = (torch.einsum(\\\"bws,bw->bs\\\", oh, meta[..., 7].float()) / cnt.clamp_min(1)).round()\\n        energy = mean * span_s[..., None]\\n        per_plane = []\\n        for pl in (0, 1, 2):\\n            ok = ((plane_s == pl) & (cnt > 0))[..., None]\\n            per_plane.append(energy.masked_fill(~ok, 0.0).amax(1))\\n        sag, cor, ax = per_plane\\n        area = torch.zeros_like(sag)\\n        area[:, OA_MED] = sag[:, OA_MED] * cor[:, OA_MED]\\n        area[:, OA_LAT] = sag[:, OA_LAT] * cor[:, OA_LAT]\\n        area[:, OA_PF] = ax[:, OA_PF] * sag[:, OA_PF]\\n        return area\\n\\n    def _decompose_syn(self, z: torch.Tensor) -> torch.Tensor:\\n        ze = z[:, EFF].detach()\\n        onehot = z.new_zeros(z.shape[1])\\n        onehot[SYN] = 1\\n        return z + (self.syn_beta * ze)[:, None] * onehot[None, :]\\n\\n    def head(self, tok: torch.Tensor, meta: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\\n        \\\"\\\"\\\"tok (N,C,d) des fenetres valides, dans l'ordre du masque. -> (B,C).\\\"\\\"\\\"\\n        b, w = mask.shape\\n        f = self.proj(tok.float())\\n        emb = (\\n            self.slot(meta[..., 0].long().clamp(0, 9))\\n            + self.view(meta[..., 1].long().clamp(0, 5))\\n            + self.geom(meta[..., 2:6])\\n        )\\n        t = f.new_zeros((b, w) + tuple(f.shape[1:]))\\n        t[mask] = f\\n        t = self.drop(t + emb[:, :, None, :])\\n        s2 = torch.einsum(\\\"bwch,ch->bwc\\\", torch.tanh(self.win_proj(t)), self.q_win).float()\\n        s2 = s2.masked_fill(~mask[:, :, None], float(\\\"-inf\\\"))\\n        a2 = s2.softmax(1).to(t.dtype)\\n        study = self.out_norm(torch.einsum(\\\"bwc,bwch->bch\\\", a2, t))\\n        e_win = (t.float() * self.ext_u[None, None]).sum(-1) + self.ext_b\\n        z = (study * self.cls_w).sum(-1).float() + self.bias\\n        area = self.extent_area(e_win, meta, mask)\\n        z = z + self.ext_gamma * self.ext_mask * torch.log1p(area / 100.0)\\n        return self._decompose_syn(z)\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        \\\"\\\"\\\"x (B, S, D, H, W) dans [0, 1] -> logits (B, 12).\\\"\\\"\\\"\\n        _b, s, d, _h, _w = x.shape\\n        if s != self.n_series:\\n            raise ValueError(f\\\"attendu {self.n_series} series, recu {s}\\\")\\n        if d > self.max_depth:\\n            raise ValueError(f\\\"D={d} > max_depth={self.max_depth}\\\")\\n        valid = x.amax(dim=(-2, -1)) > 1e-6\\n        mask, meta = self._window_meta(valid)\\n        idx = mask.reshape(-1).nonzero(as_tuple=False).view(-1)\\n        if idx.numel() == 0:\\n            return x.new_zeros(x.shape[0], self.n_targets)\\n        maps = self._encode_indexed(x, idx)\\n        tok = self.spatial_pool(maps)\\n        return self.head(tok, meta, mask)\\n\"")
_cnx_ns = {'__name__': 'kneenet_cnx_r7'}
exec(compile(_cnx_src, 'kneenet_cnx_r7.py', 'exec'), _cnx_ns)
KneeNet = _cnx_ns['KneeNet']
print('KneeNet CNX-r7 isole', KneeNet.arch_id, flush=True)


In [ ]:
import pandas as pd
import time

TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
           'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
           'Contusion', 'Fracture']


def _cap_chunk(cfg, cap):
    raw = int(cfg.get('encode_chunk', cap))
    chunk = cap if raw > cap else raw
    if chunk != raw:
        print(f'encode_chunk {raw} -> {chunk} (T4)')
    return chunk


def _expect_swa(ckpt, label, epochs):
    if str(ckpt.get('epoch')) != label:
        raise RuntimeError(ckpt.get('epoch'))
    if [int(e) for e in (ckpt.get('swa_epochs') or [])] != list(epochs):
        raise RuntimeError(ckpt.get('swa_epochs'))


def load_vit(weights_path):
    ckpt = torch.load(str(weights_path), map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    if str(cfg.get('run_name')) != 'vit_k_llrd08_320_v3b_oofmask_v20h_full_oai3':
        raise RuntimeError(cfg.get('run_name'))
    if str(cfg.get('model')) != 'kneenet_vit_k320':
        raise RuntimeError(cfg.get('model'))
    if int(cfg.get('img_size', 0)) != 320:
        raise RuntimeError(cfg.get('img_size'))
    if cfg.get('laterality'):
        raise RuntimeError('vit-k320 ne doit pas etre flipped')
    _expect_swa(ckpt, 'swa-1-2-3', [1, 2, 3])
    model = KneeNetVitE(
        n_series=int(cfg.get('n_series', 6)),
        head_dim=int(cfg.get('head_dim', 512)),
        dropout=float(cfg.get('dropout', 0.4)),
        encode_chunk=_cap_chunk(cfg, 2),
        lstm_hidden=int(cfg.get('lstm_hidden', 256)),
        pretrained=False,
    )
    model.load_state_dict(ckpt['model'], strict=True)
    model.to(DEVICE).eval()
    print('vit', cfg.get('run_name'), 'epoch', ckpt.get('epoch'),
          'gold', round(float(ckpt.get('gold_macro_auc')), 4))
    return model


def load_resnet(weights_path, cls, model_name, run_name, swa_label, swa_epochs):
    ckpt = torch.load(str(weights_path), map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    if str(cfg.get('run_name')) != run_name:
        raise RuntimeError(cfg.get('run_name'))
    if str(cfg.get('model')) != model_name:
        raise RuntimeError(cfg.get('model'))
    if cfg.get('laterality'):
        raise RuntimeError('ce run ne doit pas etre flipped')
    _expect_swa(ckpt, swa_label, swa_epochs)
    model = cls(
        backbone=cfg.get('backbone', 'resnet50'),
        n_series=int(cfg.get('n_series', 6)),
        head_dim=int(cfg.get('head_dim', 512)),
        dropout=float(cfg.get('dropout', 0.4)),
        encode_chunk=_cap_chunk(cfg, 8),
        lstm_hidden=int(cfg.get('lstm_hidden', 256)),
        pretrained=False,
    )
    model.load_state_dict(ckpt['model'], strict=True)
    model.to(DEVICE).eval()
    print(run_name, 'epoch', ckpt.get('epoch'), 'lat', cfg.get('laterality'),
          'gold', round(float(ckpt.get('gold_macro_auc')), 4))
    return model


model_vit = load_vit(WEIGHTS_VIT)
model_v20h = load_resnet(
    WEIGHTS_V20H, KneeNetV20h, 'kneenet_v20h',
    'v20h_crop145_dino_ep11_v3b_rsna_distil_main_full', 'swa-7-8-9', [7, 8, 9],
)
def load_cnx(weights_path):
    ckpt = torch.load(str(weights_path), map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    if str(cfg.get('run_name')) != 'cnx_r7_in22k_v3b_oofmask_full':
        raise RuntimeError(cfg.get('run_name'))
    if str(cfg.get('model')) != 'kneenet_cnx_r7':
        raise RuntimeError(cfg.get('model'))
    if str(cfg.get('backbone')) != 'convnext_small.fb_in22k_ft_in1k':
        raise RuntimeError(cfg.get('backbone'))
    if cfg.get('laterality'):
        raise RuntimeError('cnx ne doit pas etre flipped')
    if int(cfg.get('hidden', -1)) != 384:
        raise RuntimeError(cfg.get('hidden'))
    _expect_swa(ckpt, 'swa-7-8-9-10', [7, 8, 9, 10])
    chunk = _cap_chunk(cfg, 4)
    model = KneeNet(
        backbone=str(cfg.get('backbone')),
        n_series=int(cfg.get('n_series', 6)),
        n_targets=12,
        hidden=int(cfg.get('hidden', 384)),
        dropout=float(cfg.get('dropout', 0.1)),
        pretrained=False,
        encode_chunk=chunk,
        max_depth=int(cfg.get('max_depth', 64) or 64),
    )
    model.load_state_dict(ckpt['model'], strict=True)
    model.to(DEVICE).eval()
    print('cnx', cfg.get('run_name'), 'epoch', ckpt.get('epoch'),
          'lat', cfg.get('laterality'),
          'gold', round(float(ckpt.get('gold_macro_auc')), 4))
    return model


model_cnx = load_cnx(WEIGHTS_CNX)

test = pd.read_csv(TEST_CSV)
test_series = pd.read_csv(TEST_SERIES_CSV)
print(len(test), 'etudes | TTA x4 rotation +6 rotation -6 contraste 1.08 luminosite +0.05 | cnx vit distil | 2 GPU | sans hflip sans zoom | FOV vit/cnx 130 mm, v20h 145 mm | cnx forward fp32')
for t, w in zip(TARGETS, MIX):
    if abs(sum(w) - 1.0) > 1e-6 or min(w) < -1e-9 or len(w) != 3:
        raise RuntimeError((t, w))
    print(f'  {t:<18} vit {w[0]:.2f} v20h {w[1]:.2f} cnx {w[2]:.2f}')


def _forward(model, x):
    if DEVICE == 'cuda':
        with torch.autocast('cuda', dtype=AMP_DTYPE):
            return model(x)
    return model(x)




import math


def _rotate(x, degrees):
    """Rotation dans le plan de chaque coupe. Pas de hflip, pas de zoom."""
    if x.ndim != 5:
        raise RuntimeError(tuple(x.shape))
    deg = float(degrees)
    if abs(deg) != 6.0:
        raise RuntimeError(deg)
    b, s, d, h, w = x.shape
    ang = deg * math.pi / 180.0
    c, sn = math.cos(ang), math.sin(ang)
    theta = x.new_tensor([[c, -sn, 0.0], [sn, c, 0.0]]).unsqueeze(0)
    theta = theta.repeat(b * s * d, 1, 1)
    flat = x.reshape(b * s * d, 1, h, w)
    grid = torch.nn.functional.affine_grid(theta, flat.shape, align_corners=False)
    out = torch.nn.functional.grid_sample(
        flat, grid, mode='bilinear', padding_mode='border', align_corners=False,
    )
    return out.reshape(b, s, d, h, w)


def _contrast(x):
    """Contraste 1.08 autour de la moyenne de chaque coupe."""
    mean = x.mean(dim=(-2, -1), keepdim=True)
    return ((x - mean) * 1.08 + mean).clamp(0.0, 1.0)


def _brightness(x):
    """Luminosite +0.05, bornes [0, 1]."""
    return (x + 0.05).clamp(0.0, 1.0)


def _views(x):
    return (_rotate(x, 6), _rotate(x, -6), _contrast(x), _brightness(x))


print(
    'TTA x4 rotation +6, rotation -6, contraste 1.08, luminosite +0.05, '
    'cnx vit distil, sans hflip, sans zoom, sans echange de labels',
    flush=True,
)

N_GPU = torch.cuda.device_count() if DEVICE == 'cuda' else 1
if DEVICE == 'cuda' and N_GPU < 2:
    raise RuntimeError(('attendu 2 GPU', N_GPU))
DEV_CNX = 'cuda:0' if DEVICE == 'cuda' else 'cpu'
DEV_REST = 'cuda:1' if N_GPU >= 2 else DEV_CNX
print('gpus visibles', N_GPU, '| cnx', DEV_CNX, '| reste', DEV_REST, flush=True)
model_cnx.to(DEV_CNX)
model_vit.to(DEV_REST)
model_v20h.to(DEV_REST)


def _probs_cnx(x_224):
    with torch.no_grad():
        if str(DEV_CNX).startswith('cuda'):
            torch.cuda.set_device(DEV_CNX)
        x_cnx = x_224.to(DEV_CNX).float()
        with torch.autocast('cuda', enabled=False):
            ps = [torch.sigmoid(model_cnx(v).float()) for v in _views(x_cnx)]
        return (sum(ps) / 4.0).float().cpu()


def _probs_rest(x_vit, x_h):
    with torch.no_grad():
        if str(DEV_REST).startswith('cuda'):
            torch.cuda.set_device(DEV_REST)
        x_vit = x_vit.to(DEV_REST)
        x_h = x_h.to(DEV_REST)
        def _avg(model, x):
            ps = [torch.sigmoid(_forward(model, v).float()) for v in _views(x)]
            return sum(ps) / 4.0
        p_vit = _avg(model_vit, x_vit)
        p_v20h = _avg(model_v20h, x_h)
        return p_vit.float().cpu(), p_v20h.float().cpu()


_POOL = None
if DEV_REST != DEV_CNX:
    from concurrent.futures import ThreadPoolExecutor
    _POOL = ThreadPoolExecutor(max_workers=2)
rows = []
t0 = time.time()
with torch.no_grad():
    for i, uid in enumerate(test.StudyInstanceUID):
        _SERIES_CACHE.clear()
        srows = test_series[test_series.StudyInstanceUID == uid]
        sdir = TEST_SERIES_DIR / uid
        CROP_MM = 130.0
        vol_224 = study_to_tensor(sdir, srows, n_series=6, laterality=False, img_size=224)
        vol_vit = study_to_tensor(sdir, srows, n_series=6, laterality=False, img_size=320)
        _SERIES_CACHE.clear()
        CROP_MM = 145.0
        vol_h = study_to_tensor(sdir, srows, n_series=6, laterality=False, img_size=224)
        CROP_MM = 130.0
        if vol_vit.shape[-1] != 320 or vol_224.shape[-1] != 224 or vol_h.shape[-1] != 224:
            raise RuntimeError((vol_vit.shape, vol_224.shape, vol_h.shape))
        x_224_cpu = torch.from_numpy(vol_224).unsqueeze(0)
        x_vit_cpu = torch.from_numpy(vol_vit).unsqueeze(0)
        x_h_cpu = torch.from_numpy(vol_h).unsqueeze(0)
        if _POOL is None:
            p_cnx = _probs_cnx(x_224_cpu)
            p_vit, p_v20h = _probs_rest(x_vit_cpu, x_h_cpu)
        else:
            fa = _POOL.submit(_probs_cnx, x_224_cpu)
            fb = _POOL.submit(_probs_rest, x_vit_cpu, x_h_cpu)
            p_cnx = fa.result()
            p_vit, p_v20h = fb.result()
        w = torch.tensor(MIX, dtype=torch.float32)
        prob = (
            w[:, 0] * p_vit + w[:, 1] * p_v20h + w[:, 2] * p_cnx
        ).numpy()[0]
        rows.append([uid] + prob.tolist())
        if (i + 1) % 50 == 0:
            print(f'{i+1}/{len(test)} | {(time.time()-t0)/60:.1f} min', flush=True)

if _POOL is not None:
    _POOL.shutdown(wait=True)
del _probs_cnx, _probs_rest, _POOL
pred = pd.DataFrame(rows, columns=['StudyInstanceUID'] + TARGETS)
pred.head()


In [ ]:
# --- submission.csv au format exact de sample_submission --------------------
# Obligatoire pour le scoring Kaggle : ecrire /kaggle/working/submission.csv
from pathlib import Path as _Out

sample = pd.read_csv(SAMPLE_SUB)
sub = pred.set_index('StudyInstanceUID').reindex(sample.StudyInstanceUID).reset_index()
sub = sub[list(sample.columns)]
prior = sub[TARGETS].mean()
sub[TARGETS] = sub[TARGETS].fillna(prior)

out_dir = _Out('/kaggle/working') if _Out('/kaggle/working').exists() else _Out('.')
out_csv = out_dir / 'submission.csv'
sub.to_csv(out_csv, index=False)
cwd_csv = _Out('submission.csv')
if cwd_csv.resolve() != out_csv.resolve():
    sub.to_csv(cwd_csv, index=False)

print('ecrit', out_csv.resolve(), '| lignes', len(sub), '| existe', out_csv.exists())
print(sub.shape, '| colonnes OK :', list(sub.columns) == list(sample.columns))
print(sub[TARGETS].describe().loc[['mean', 'std', 'min', 'max']].round(3).to_string())
assert out_csv.exists() and out_csv.stat().st_size > 0, 'submission.csv manquant'
assert not sub[TARGETS].isna().any().any(), 'NaN dans la submission !'
assert list(sub.columns) == list(sample.columns), 'colonnes != sample_submission'
sub.head()
